In [1]:
from model import SoundSequenceAnalyzer, CustomDataset
import torch
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
from torch.utils.data import  DataLoader


In [6]:
def process_long_sequence(model, X, difficulty, window_size, device="cpu"):
    """
    Slices an audio track sequentially into window_size segments,
    evaluates them via the model, and joins the final predictions.
    """
    model.eval()
    model.to(device)

    # Ensure standard batch dimensions exist [B, Features, Time]
    if X.dim() == 2:
        X = X.unsqueeze(0)
    if difficulty.dim() == 2:
        difficulty = difficulty.unsqueeze(0)

    total_frames = X.shape[-1]
    num_chunks = total_frames // window_size

    # Storage arrays for collecting outputs across loops
    all_is_object = []
    all_pi_logits = []
    all_mu = []
    all_log_sigma = []
    all_obj_types = []
    all_obj_attrs = []
    all_curve_types = []
    all_anchor_count = []
    all_anchor_coords = []
    with torch.no_grad():
        for i in range(num_chunks):
            start = i * window_size
            end = start + window_size

            # Slice step
            X_chunk = X[..., start:end].to(device)
            diff_chunk = difficulty[:, start:end, :].to(device)

            # Evaluate forward pass
            outputs = model(X_chunk, diff_chunk)

            # Accommodates both 3-output and 4-output model versions cleanly

            is_obj_pred, mdn_out, obj_type_pred, obj_attr_pred, curve_type, anchor_count, anchor_coords = outputs
            all_obj_attrs.append(obj_attr_pred.cpu())


            pi_logits, mu, log_sigma = mdn_out

            # Keep on CPU to preserve VRAM during loop steps
            all_is_object.append(is_obj_pred.cpu())
            all_pi_logits.append(pi_logits.cpu())
            all_mu.append(mu.cpu())
            all_log_sigma.append(log_sigma.cpu())
            all_obj_types.append(obj_type_pred.cpu())
            all_curve_types.append(curve_type.cpu())
            all_anchor_coords.append(anchor_coords.cpu())
            all_anchor_count.append(anchor_count.cpu())

    # Concatenate chunk blocks back into standard continuous timelines (dim 1)
    full_is_object = torch.cat(all_is_object, dim=1).to(device)
    full_pi_logits = torch.cat(all_pi_logits, dim=1).to(device)
    full_mu = torch.cat(all_mu, dim=1).to(device)
    full_log_sigma = torch.cat(all_log_sigma, dim=1).to(device)
    full_obj_types = torch.cat(all_obj_types, dim=1).to(device)
    full_curve_types = torch.cat(all_curve_types, dim=1).to(device)
    full_anchor_count = torch.cat(all_anchor_count, dim=1).to(device)
    full_anchor_coords = torch.cat(all_anchor_coords, dim=1).to(device)
    mdn_recombined = (full_pi_logits, full_mu, full_log_sigma)

    if all_obj_attrs:
        full_obj_attrs = torch.cat(all_obj_attrs, dim=1)
        return ((full_is_object), (mdn_recombined), (full_obj_types, full_obj_attrs, full_curve_types, full_anchor_count, full_anchor_coords))

    return full_is_object, mdn_recombined, full_obj_types

In [ ]:

def evaluate_model(model, val_loader, window_size, device="cpu"):
    """
    Performs batch-by-batch validation evaluation across all prediction heads.

    Args:
        model: Trained SoundSequenceAnalyzer network.
        val_loader: DataLoader yielding (X, targets, difficulty) batches.
        device: 'cuda' or 'cpu'.
    """
    model.eval()
    model.to(device)

    # Storage for Onset (is_object) detection metrics
    all_onset_targets = []
    all_onset_preds = []

    # Storage for Spatial (coordinate) error tracking
    coord_errors_pixels = [] # L1 distance in osu! pixels
    coord_mse_accum = 0.0
    total_valid_objects = 0

    # Storage for Category (obj_type) classification metrics
    all_type_targets = []
    all_type_preds = []

    print("Starting model evaluation on validation split...")

    with torch.no_grad():
        for batch_idx, (X, targets, difficulty) in enumerate(val_loader):
            X = X.to(device)
            targets = targets.to(device)
            difficulty = difficulty.to(device)

            # 1. Forward Pass Through Your Upgraded Architecture
            # Unpack all heads (including the MDN and curve heads we added)
            
            outputs = process_long_sequence(model, X, difficulty, window_size, device=device)
            is_object_pred = outputs[0]
            mdn_out = outputs[1]
            obj_type_pred = outputs[2]

            pi_logits, mu, log_sigma = mdn_out

            # --- Head 1: is_object Evaluation ---
            # Ground truth binary targets shape: [B, Time]
            onset_target = targets[:, :, 0].cpu().numpy().flatten()
            # Predicted sigmoid logits thresholded at 0.5 to get binary decisions
            onset_pred = (torch.sigmoid(is_object_pred).squeeze(-1) > 0.5).cpu().numpy().flatten()

            all_onset_targets.extend(onset_target)
            all_onset_preds.extend(onset_pred)

            # --- Head 2: Coordinate (MDN) Evaluation ---
            # Create boolean mask matching frames where objects exist in ground truth
            coord_mask = (targets[:, :, 0] == 1.0)

            if coord_mask.any():
                # Extract ground-truth normalized coordinates (x, y)
                gt_coords = targets[:, :, 1:3][coord_mask] # Shape: [N_Objects, 2]

                # Sample or select from the MDN output for evaluation
                # To test coordinate accuracy fairly, we use the center (mu) of the highest-weight component
                best_component = pi_logits[coord_mask].argmax(dim=-1) # Shape: [N_Objects]
                idx = best_component.unsqueeze(-1).unsqueeze(-1).expand(best_component.size(0), 1, 2)
                pred_coords = mu[coord_mask].gather(-2, idx).squeeze(-2) # Shape: [N_Objects, 2]

                # Denormalize coordinates to standard osu! canvas space pixels for interpretable metrics
                # Playfield boundaries: X is 0 to 512, Y is 0 to 384
                scale = torch.tensor([512.0, 384.0], device=device)
                gt_coords_pixels = gt_coords * scale
                pred_coords_pixels = pred_coords * scale

                # Calculate Euclidean/L1 distance error per object
                distances = torch.sqrt(torch.sum((pred_coords_pixels - gt_coords_pixels) ** 2, dim=-1))
                coord_errors_pixels.extend(distances.cpu().numpy())

                # Track raw normalized MSE for loss verification
                mse_loss = torch.sum((pred_coords - gt_coords) ** 2).item()
                coord_mse_accum += mse_loss
                total_valid_objects += coord_mask.sum().item()

                # --- Head 3: Object Type Evaluation ---
                # Slice target mapping: [3:6] corresponds to Circle, Slider, Spinner
                gt_types = torch.argmax(targets[:, :, 3:6][coord_mask], dim=-1).cpu().numpy()
                pred_types = torch.argmax(obj_type_pred[coord_mask], dim=-1).cpu().numpy()

                all_type_targets.extend(gt_types)
                all_type_preds.extend(pred_types)

    # ==========================================
    # METRIC COMPILATION AND REPORT GENERATION
    # ==========================================
    print("\n" + "="*50)
    print("              EVALUATION REPORT             ")
    print("="*50)

    # 1. Timeline Onset Placement Results
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_onset_targets, all_onset_preds, average='binary'
    )
    print(f"\n[1/3] Rhythmical Alignment (is_object):")
    print(f"  • Precision : {precision:.4f}")
    print(f"  • Recall    : {recall:.4f}")
    print(f"  • F1-Score  : {f1:.4f}")

    # 2. Geometric Coordinate Tracking Results
    if total_valid_objects > 0:
        mean_pixel_error = np.mean(coord_errors_pixels)
        median_pixel_error = np.median(coord_errors_pixels)
        normalized_mse = coord_mse_accum / (total_valid_objects * 2)

        print(f"\n[2/3] Canvas Coordinate Accuracy (Masked to valid objects):")
        print(f"  • Mean Displacement   : {mean_pixel_error:.2f} osu! pixels")
        print(f"  • Median Displacement : {median_pixel_error:.2f} osu! pixels")
        print(f"  • Normalized MSE      : {normalized_mse:.6f}")
    else:
        print("\n[2/3] Canvas Coordinate Accuracy: No active target frames found to evaluate spatial metrics.")

    # 3. Object Type Classification Results
    if len(all_type_targets) > 0:
        print(f"\n[3/3] Structural Object Classification (obj_type):")
        target_names = ['Circle', 'Slider', 'Spinner']

        # Display dense precision, recall, f1 grid across all 3 classes
        print(classification_report(all_type_targets, all_type_preds, target_names=target_names, digits=4))

        # Calculate raw breakdown matrix
        cm = confusion_matrix(all_type_targets, all_type_preds)
        print("Confusion Matrix:")
        print("                 Predicted")
        print("              Circle  Slider  Spinner")
        print(f"Actual Circle   {cm[0,0]:<7} {cm[0,1]:<7} {cm[0,2]:<7}")
        # Safeguard print statements if validation batch lacks rare spinner samples
        s_act = cm[1,0] if cm.shape[0] > 1 else 0
        s_sl  = cm[1,1] if cm.shape[0] > 1 else 0
        s_sp  = cm[1,2] if cm.shape[0] > 1 else 0
        print(f"       Slider   {s_act:<7} {s_sl:<7} {s_sp:<7}")
        sp_act = cm[2,0] if cm.shape[0] > 2 else 0
        sp_sl  = cm[2,1] if cm.shape[0] > 2 else 0
        sp_sp  = cm[2,2] if cm.shape[0] > 2 else 0
        print(f"       Spinner  {sp_act:<7} {sp_sl:<7} {sp_sp:<7}")
    else:
        print("\n[3/3] Structural Object Classification: No object categories to calculate.")

    print("="*50)

    # Return structured dict for logging tracking hooks (e.g., Weights & Biases)
    torch.cuda.empty_cache()
    return {
        "onset_f1": f1,
        "coord_mean_pixel_error": mean_pixel_error if total_valid_objects > 0 else None,
        "type_macro_f1": np.mean(precision_recall_fscore_support(all_type_targets, all_type_preds, average='macro')[2]) if len(all_type_targets) > 0 else None
    }



In [4]:

window_size = 256
n_heads = 16
freq_vec_len = 128
model = SoundSequenceAnalyzer(time_window_size=window_size, freq_vec_len=freq_vec_len, num_heads=n_heads)
state_dict = torch.load('w_1k_256w_16h_128v.pth', weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_dataset, _ = CustomDataset.split(
    train_ratio=1,
    seed=42,
    window_size=window_size,
    debug=False
)


dataloader = DataLoader(train_dataset, batch_size=128, shuffle=False)
evaluate_model(model, dataloader, window_size, device)

Starting model evaluation on validation split...



KeyboardInterrupt

